# 太陽能板瑕疵檢測與成因說明

長榮大學資訊工程學系｜2026.12.05

這次讓兩個模型接力，完成一次太陽能板巡檢。

| 第一段：PRB-FPN 物件偵測 | 第二段：Qwen 語言模型 |
|---|---|
| 輸入 EO／IR 圖片 | 輸入偵測結果整理出的文字線索 |
| 找出面板、破裂與熱點的位置 | 寫出異常的可能成因 |

先一起操作 **case01**，再換一組自己練習。每完成一段，先看結果、說說你的判斷，再繼續。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andy6804tw/speech-archive/blob/main/2026/20261205-%E9%95%B7%E6%A6%AE%E5%A4%A7%E5%AD%B8%E8%B3%87%E5%B7%A5%E7%B3%BB-AI%E5%AF%A6%E4%BD%9C%E8%88%87%E5%AD%B8%E9%95%B7%E7%B6%93%E9%A9%97%E5%88%86%E4%BA%AB/notebooks/%E5%A4%AA%E9%99%BD%E8%83%BD%E6%9D%BF%E7%91%95%E7%96%B5%E6%AA%A2%E6%B8%AC/%E5%A4%AA%E9%99%BD%E8%83%BD%E6%9D%BF%E7%91%95%E7%96%B5%E6%AA%A2%E6%B8%AC%E8%88%87%E6%88%90%E5%9B%A0%E8%AA%AA%E6%98%8E.ipynb)


## 1. 準備實作環境

在 Colab 的「執行階段 → 變更執行階段類型」選擇 **T4 GPU**，再依序執行下面兩格。

程式會準備課程素材、套件與官方模型。第一次需要等幾分鐘，模型解壓後約 **1.13 GB**，無需 API 金鑰。
這次使用已訓練好的模型進行**推論**；沒有 GPU 也可以用 CPU，但速度較慢。

In [ ]:
# @title 讀取實作素材
from pathlib import Path
from urllib.request import urlopen
import hashlib
import sys
import zipfile

素材檔 = Path("太陽能板實作素材.zip")
素材網址 = 'https://raw.githubusercontent.com/andy6804tw/speech-archive/main/2026/20261205-%E9%95%B7%E6%A6%AE%E5%A4%A7%E5%AD%B8%E8%B3%87%E5%B7%A5%E7%B3%BB-AI%E5%AF%A6%E4%BD%9C%E8%88%87%E5%AD%B8%E9%95%B7%E7%B6%93%E9%A9%97%E5%88%86%E4%BA%AB/notebooks/%E5%A4%AA%E9%99%BD%E8%83%BD%E6%9D%BF%E7%91%95%E7%96%B5%E6%AA%A2%E6%B8%AC/%E5%A4%AA%E9%99%BD%E8%83%BD%E6%9D%BF%E5%AF%A6%E4%BD%9C%E7%B4%A0%E6%9D%90.zip'
素材版本 = "df81b5051e1e3dadb71a9177f84f36bd6487f9fd948a11b5dccfcec7df931bbf"

素材網址 += "?v=" + 素材版本[:12]

if not 素材檔.exists() or hashlib.sha256(素材檔.read_bytes()).hexdigest() != 素材版本:
    print("正在從 GitHub 讀取實作素材……")
    with urlopen(素材網址, timeout=60) as response:
        素材內容 = response.read()
    if hashlib.sha256(素材內容).hexdigest() != 素材版本:
        raise ValueError("素材下載不完整，請重新執行這格。")
    素材檔.write_bytes(素材內容)

教材目錄 = (Path.cwd() / "solar_classroom").resolve()
with zipfile.ZipFile(素材檔) as 壓縮檔:
    壓縮檔.extractall(教材目錄)
sys.path.insert(0, str(教材目錄))

sys.modules.pop("solar_classroom", None)
from solar_classroom import SolarLab
lab = SolarLab(教材目錄)
print("素材已就緒：case01 示範，case02～case10 自選練習。")

In [ ]:
# @title 準備套件、官方模型與課堂工具（第一次需要等待）
lab.setup_environment()
lab.prepare_models()

"""Notebook scaffolding: files, logs and the official prompt formatter."""
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

# These names are initialized by the first preparation cell in the notebook.
PYTHON = str(lab.python)
BASELINE = lab.base
DETECTOR_WEIGHTS = lab.models / "best.pt"
QWEN_WEIGHTS = lab.models / "qwen_reason"
CANDIDATES_FILE = 教材目錄 / "candidates.json"
sys.path.insert(0, str(BASELINE / "reason"))
from common import ANOMALY, bbox, describe, mod_of, paired_key


def 準備案例(case_id):
    if case_id not in lab.cases:
        raise ValueError("請從選單選擇 case01～case10。")
    area = lab.results / case_id
    test_root = area / "test"
    output_dir = area / "run"
    output_dir.mkdir(parents=True, exist_ok=True)
    for mod in ("EO", "IR"):
        destination = test_root / mod / "image" / f"{case_id}.jpg"
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(教材目錄 / "cases" / case_id / f"{mod}.jpg", destination)
    return test_root, output_dir


def 執行官方程式(command, stage):
    if OUTPUT_DIR.parent.name != CASE_ID:
        raise RuntimeError("案例已改變，請先重新執行第 2 節的案例選單。")
    if stage == "物件偵測":
        CONFIDENCE_VALUE = float(CONFIDENCE)
        if not 0.05 <= CONFIDENCE_VALUE <= 0.95:
            raise ValueError("課堂信心門檻請設在 0.05～0.95。")
        # Re-running detection invalidates any explanation from the previous run.
        for old in (OUTPUT_DIR / "detections.json", OUTPUT_DIR / "submission.json"):
            old.unlink(missing_ok=True)
    elif not (OUTPUT_DIR / "detections.json").is_file():
        raise RuntimeError("請先完成第 3 節的物件偵測，再執行文字說明。")
    env = dict(lab.env)
    if lab.device == "cpu":
        env["CUDA_VISIBLE_DEVICES"] = "-1"
    log = OUTPUT_DIR / f"{stage}.log"
    start = time.monotonic()
    print(f"正在執行{stage}：{CASE_ID}……", flush=True)
    with log.open("w", encoding="utf-8") as stream:
        result = subprocess.run([str(x) for x in command], cwd=BASELINE, env=env,
                                stdout=stream, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError("這一步沒有完成，執行紀錄如下：\n" + log.read_text(encoding="utf-8")[-6000:])
    print(f"{stage}完成（{time.monotonic() - start:.1f} 秒）。", flush=True)


def 建立文字線索(detections):
    """Use exactly the formatter and object order used by infer_submission.py."""
    objects = {key: [dict(obj, box=bbox(obj["points"])) for obj in entry["objects"]]
               for key, entry in detections.items()}
    jobs = []
    for key, objs in objects.items():
        entry = detections[key]
        partner_key = paired_key(key)
        partner = detections.get(partner_key)
        for index, obj in enumerate(objs):
            if obj["label"] in ANOMALY:
                text = describe(obj, objs, entry["width"], entry["height"], mod_of(key),
                                objects.get(partner_key), partner["width"] if partner else None,
                                partner["height"] if partner else None)
                jobs.append({"key": key, "index": index, "label": obj["label"], "prompt": text})
    return jobs

print("準備完成！從第 2 節開始觀察圖片。")

## 2. 先用眼睛找一找

每個案例有兩張對應的圖片：

- **EO 可見光**：觀察面板外觀。
- **IR 熱影像**：觀察熱分布差異；這份資料用灰階呈現。

先選 **case01**。執行後想一想：哪片面板外觀不一樣？熱影像哪裡比較亮？

In [ ]:
# @title 選擇案例，再按 ▶
CASE_ID = "case01" # @param ["case01", "case02", "case03", "case04", "case05", "case06", "case07", "case08", "case09", "case10"]

TEST_ROOT, OUTPUT_DIR = 準備案例(CASE_ID)
print(f"目前案例：{CASE_ID}")
lab.show_inputs(CASE_ID)

## 3. 讓 PRB-FPN 找出物件位置

現在把這兩張圖片交給 **PRB-FPN**。這一段會處理熱影像、偵測物件，再整理結果。

程式中的 `--test-root` 指定圖片、`--det-weights` 指定已訓練的模型；`CONFIDENCE` 是保留偵測框的信心門檻。
第一次維持 **0.25**，先看看結果。這個分數不等於模型的正確率。

In [ ]:
# @title 執行物件偵測
CONFIDENCE = 0.25 # @param {type:"slider", min:0.05, max:0.95, step:0.05}

detect_command = [
    PYTHON, BASELINE / "pipeline.py", "ir-norm", "detect", "dets",
    "--test-root", TEST_ROOT,               # 這組 EO／IR 圖片
    "--det-weights", DETECTOR_WEIGHTS,      # PRB-FPN 模型權重
    "--work", OUTPUT_DIR, "--device", lab.device,
    "--conf", f"solar_panel={CONFIDENCE},crack={CONFIDENCE},hotspot={CONFIDENCE}",
]
執行官方程式(detect_command, "物件偵測")

### 看看框的位置，再看看資料

執行下一格，先看框有沒有畫在你剛才注意到的地方。

**藍框：正常面板　紅框：破裂　橘框：熱點**

偵測結果會存成 `detections.json`。程式把它讀進 `detections`，裡面記錄了圖片尺寸、物件類別、信心分數與框的座標。

In [ ]:
import json

detections_file = OUTPUT_DIR / "detections.json"
detections = json.loads(detections_file.read_text(encoding="utf-8"))

lab.show_detections(CASE_ID)

In [ ]:
import pandas as pd

異常框 = []
for image_name, entry in detections.items():
    for obj in entry["objects"]:
        if obj["label"] in ("crack", "hotspot"):
            異常框.append({"影像": image_name.split("/")[1], "類別": obj["label"],
                         "信心分數": round(obj["conf"], 2),
                         "左上角": obj["points"][0], "右下角": obj["points"][2]})

display(pd.DataFrame(異常框, columns=["影像", "類別", "信心分數", "左上角", "右下角"]))

圖和表格描述的是同一批結果。座標表示框在圖片上的位置，單位是像素；左上角是起點，往右是 x 增加，往下是 y 增加。

**先停一下：模型找到的破裂與熱點，是你原本猜的位置嗎？** 若表格是空的，表示這次沒有留下異常框，可以繼續看看下一段如何處理。

## 4. 把偵測結果變成文字線索

PRB-FPN 已經告訴我們「哪裡有什麼」。接下來使用官方的文字整理方式，將**類別、位置、數量與另一張影像的結果**整理成 Qwen 的輸入。

下面顯示一個異常的實際輸入文字。找找看：它提到哪種瑕疵？有沒有引用另一張影像的線索？

In [ ]:
from common import SYSTEM_PROMPT

文字線索 = 建立文字線索(detections)
print("角色與任務：", SYSTEM_PROMPT)

if 文字線索:
    範例 = next((item for item in 文字線索 if item["label"] == "hotspot"), 文字線索[0])
    print("\n偵測線索：\n" + 範例["prompt"])
else:
    print("\n這次沒有偵測到異常，因此沒有要說明的瑕疵。")

**這份 baseline 的 Qwen 是文字語言模型（LLM），這一段讀的是文字線索。**
下一格的官方成因程式會讀取同一份 `detections.json`，使用相同的整理方式，為每一個異常準備文字輸入。這裡先觀察輸入內容。

## 5. 讓 Qwen 說明可能原因

執行下一格。`--dets` 是剛才的偵測結果，`--model` 是主辦方已微調的 Qwen 模型，`--out` 是最後結果要存放的位置。

這次沿用官方的 `snap` 模式：模型先生成，再對齊到候選句，讓輸出保持完整。

In [ ]:
submission_file = OUTPUT_DIR / "submission.json"

reason_command = [
    PYTHON, BASELINE / "reason" / "infer_submission.py",
    "--dets", detections_file,             # 接上第一段的輸出
    "--model", QWEN_WEIGHTS,               # 已微調的 Qwen 模型
    "--candidates", CANDIDATES_FILE, "--mode", "snap",
    "--out", submission_file,
]
執行官方程式(reason_command, "成因說明")

In [ ]:
results = json.loads(submission_file.read_text(encoding="utf-8"))

成因說明 = []
for image_name, objects in results.items():
    for obj in objects:
        if "reason" in obj:
            成因說明.append({"影像": image_name.split("/")[1],
                           "類別": obj["label"], "可能成因": obj["reason"]})

if 成因說明:
    display(pd.DataFrame(成因說明))
else:
    print("這次沒有異常框，因此沒有產生成因文字。")

第一段輸出了**物件類別與位置**，第二段替異常物件加上了 `reason` 成因文字。

**這段說明有剛才的偵測線索支持嗎？** 說明是模型的推測，還需要對照圖片與標註檢查。

## 6. 對照標註，換你做一次

執行下一格，**左邊看模型預測，右邊看官方標註**；上排是 EO，下排是 IR。
同一張圖片以相同大小並排，先看看：框的位置有沒有偏移？模型有沒有漏抓或多抓？

同一個異常若能依位置配對，兩邊會使用相同編號，並在下方並列成因文字。
標示 **P** 的是未配對的模型框；未配到模型框的官方異常也會保留，請回到圖片確認。
官方可能沿面板畫多邊形，模型畫的是矩形框；配對只是方便對照，不代表模型判定正確。

**成因不必逐字相同，但描述的異常與原因是否一致？有沒有加入影像無法支持的推測？**

In [ ]:
lab.show_reference(CASE_ID)

### 選一組，再走過兩段模型

回到**第 2 節的案例選單**換一組，從那格開始往下執行。先跑物件偵測、觀察結果，再跑成因說明。

| 想觀察什麼？ | 可以選的案例 |
|---|---|
| 正常面板，看看是否產生成因 | case02、case10 |
| 外觀破裂與熱點的對照 | case03、case05 |
| 局部破損或較小的區域 | case04、case08、case09 |
| 進一步檢查漏框與文字合理性 | case06、case07 |

雙擊這個文字區塊，留下你的觀察：

- 我選的案例：
- 我先看到的異常：
- 模型找到什麼？有沒有漏掉或多框？
- Qwen 的說明，能從哪些線索看出來？

有餘裕可以回到 **case01**，把第 3 節的信心門檻改成 **0.90**，從該格往下重跑。觀察框和後續成因是否改變；提高門檻也可能把真實異常一起濾掉。

## 7. 把結果帶走

最後的 `submission.json` 保留物件類別、位置座標，以及異常物件的成因文字。
執行下一格，下載**目前案例**的練習結果。

In [ ]:
結果檔 = lab.download_result(CASE_ID)

### 今天完成的流程

**EO／IR 圖片 → PRB-FPN 偵測類別與位置 → 整理文字線索 → Qwen 生成可能成因 → 對照結果**

兩段模型各有分工，文字說明也會受到前面偵測結果的影響。
這次使用已訓練模型；要讓模型學習新的特徵，才需要另行準備標註資料並訓練或微調。

這 10 組案例來自競賽訓練集，適合練習操作，不能用來估算未見資料的準確率。下載檔只包含目前案例，不是完整競賽提交檔。

資料與模型：[競賽頁面](https://www.aidea-web.tw/topic/3c8ec48b-0dff-4b14-9b5e-d91bbb98c699?lang=zh)・[主辦方資料](https://drive.google.com/drive/folders/1PO9jYBlQXbkAgbp-xTySJWQdnBXF2wGz)。原始授權與出處保留於素材 ZIP 的 `來源與使用說明.md`。